## Code for Install

In [1]:
! pip install yfinance

In [2]:
!pip install ta

In [3]:
! pip install pandas

## Code for VRTX

In [4]:
import pandas as pd
import yfinance as yf
from ta.trend import EMAIndicator

favourites = ['VRTX','REGN','BCRX','INO', 'NRXP', 'ARWR']
company_name = 'NA'
def closing_year(code):
    global company_name
    results = yf.download(code, period="1y", multi_level_index=False)
    ticker = yf.Ticker(code)
    # Get company information
    company_name = ticker.info.get('longName')
    return results
    
def closing_month(code):
    global company_name
    results = yf.download(code, period="1mo", multi_level_index=False)
    ticker = yf.Ticker(code)
    # Get company information
    company_name = ticker.info.get('longName')
    return results
    
# This is Fetch Data for VRTX company name
df = closing_year(favourites[0]) ## If want Know other company name can change the number for 0-5.
# df = yf.download("VRTX", period="1y", multi_level_index=False)
print(company_name)

[*********************100%***********************]  1 of 1 completed


Vertex Pharmaceuticals Incorporated


In [5]:
df

,Close,High,Low,Open,Volume
Date,,,,,
2025-05-07,434.820007,449.000000,432.290009,447.519989,3996700
2025-05-08,429.600006,432.140015,423.399994,430.170013,2940800
2025-05-09,424.989990,434.929993,424.799988,430.220001,1642300
2025-05-12,439.369995,441.100006,423.200012,425.299988,2095100
2025-05-13,432.000000,442.119995,429.540009,437.529999,2645600
...,...,...,...,...,...
2026-05-01,423.920013,427.380005,421.540009,425.019989,796900
2026-05-04,429.850006,430.130005,422.109985,423.750000,1418900
2026-05-05,424.359985,433.940002,412.269989,431.989990,1905900


In [6]:
df['EMA50'] = EMAIndicator(df['Close'], window=18).ema_indicator()
df

,Close,High,Low,Open,Volume,EMA50
Date,,,,,,
2025-05-07,434.820007,449.000000,432.290009,447.519989,3996700,NaN
2025-05-08,429.600006,432.140015,423.399994,430.170013,2940800,NaN
2025-05-09,424.989990,434.929993,424.799988,430.220001,1642300,NaN
2025-05-12,439.369995,441.100006,423.200012,425.299988,2095100,NaN
2025-05-13,432.000000,442.119995,429.540009,437.529999,2645600,NaN
...,...,...,...,...,...,...
2026-05-01,423.920013,427.380005,421.540009,425.019989,796900,434.358338
2026-05-04,429.850006,430.130005,422.109985,423.750000,1418900,433.883777
2026-05-05,424.359985,433.940002,412.269989,431.989990,1905900,432.881273


In [7]:
df['EMA200'] = df['Close'].ewm(span=18, adjust=False).mean()
df

,Close,High,Low,Open,Volume,EMA50,EMA200
Date,,,,,,,
2025-05-07,434.820007,449.000000,432.290009,447.519989,3996700,NaN,434.820007
2025-05-08,429.600006,432.140015,423.399994,430.170013,2940800,NaN,434.270534
2025-05-09,424.989990,434.929993,424.799988,430.220001,1642300,NaN,433.293634
2025-05-12,439.369995,441.100006,423.200012,425.299988,2095100,NaN,433.933251
2025-05-13,432.000000,442.119995,429.540009,437.529999,2645600,NaN,433.729751
...,...,...,...,...,...,...,...
2026-05-01,423.920013,427.380005,421.540009,425.019989,796900,434.358338,434.358338
2026-05-04,429.850006,430.130005,422.109985,423.750000,1418900,433.883777,433.883777
2026-05-05,424.359985,433.940002,412.269989,431.989990,1905900,432.881273,432.881273


In [8]:
data = df.filter(['Close'])
data

,Close
Date,
2025-05-07,434.820007
2025-05-08,429.600006
2025-05-09,424.989990
2025-05-12,439.369995
2025-05-13,432.000000
...,...
2026-05-01,423.920013
2026-05-04,429.850006
2026-05-05,424.359985


In [9]:
# Calculate 50 and 200-day EMAs
data['EMA50'] = data['Close'].ewm(span=50, adjust=False).mean()
data['EMA200'] = data['Close'].ewm(span=200, adjust=False).mean()

# Define the crossover logic
# Signal is 1 when EMA50 > EMA200, else 0
data['Signal'] = 0.0
data.loc[data['EMA50'] > data['EMA200'], 'Signal'] = 1.0

# Find the exact crossover points
# Position is the difference in signals (1 = Golden Cross, -1 = Death Cross)
# Filter days with buy signal (==1)
data['Position'] = data['Signal'].diff()

# Extract only the dates where a crossover happened
crossovers = data[data['Position'] != 0].dropna()
print(crossovers[['Close', 'EMA50', 'EMA200', 'Position']])

                 Close       EMA50      EMA200  Position
Date                                                    
2025-05-20  447.179993  434.801459  434.779839       1.0
2025-08-13  395.920013  438.468692  439.279894      -1.0
2025-12-08  442.040009  425.482445  425.381479       1.0
2026-05-07  427.345001  443.195764  443.491407      -1.0


In [10]:
# Filter days with buy signal
buys = crossovers[crossovers['Signal'] == 1]
print(buys.head())

                 Close       EMA50      EMA200  Signal  Position
Date                                                            
2025-05-20  447.179993  434.801459  434.779839     1.0       1.0
2025-12-08  442.040009  425.482445  425.381479     1.0       1.0


## Code for REGN

In [11]:
import pandas as pd
import yfinance as yf
from ta.trend import EMAIndicator

favourites = ['VRTX','REGN','BCRX']
company_name = 'NA'
def closing_year(code):
    global company_name
    results = yf.download(code, period="1y", multi_level_index=False)
    ticker = yf.Ticker(code)
    # Get company information
    company_name = ticker.info.get('longName')
    return results
    
def closing_month(code):
    global company_name
    results = yf.download(code, period="1mo", multi_level_index=False)
    ticker = yf.Ticker(code)
    # Get company information
    company_name = ticker.info.get('longName')
    return results
    
# This is Fetch Data for VRTX company name
df = closing_year(favourites[1]) ## If want Know other company name can change the number for 0-5.
# df = yf.download("VRTX", period="1y", multi_level_index=False)
print(company_name)

[*********************100%***********************]  1 of 1 completed


Regeneron Pharmaceuticals, Inc.


In [12]:
df

,Close,High,Low,Open,Volume
Date,,,,,
2025-05-07,557.881897,565.042784,553.336723,558.737209,1140500
2025-05-08,544.693970,549.567368,517.671627,544.902850,2709200
2025-05-09,524.912109,549.835933,524.265617,546.016772,1499000
2025-05-12,572.502075,574.849237,532.381253,540.864931,1898300
2025-05-13,571.040039,571.925217,555.415445,568.344804,1149900
...,...,...,...,...,...
2026-05-01,701.419983,714.479980,700.260010,703.020020,729100
2026-05-04,709.210022,711.549988,699.750000,702.559998,554100
2026-05-05,702.270020,715.619995,699.229980,710.729980,642700


In [13]:
df['EMA50'] = EMAIndicator(df['Close'], window=18).ema_indicator()
df

,Close,High,Low,Open,Volume,EMA50
Date,,,,,,
2025-05-07,557.881897,565.042784,553.336723,558.737209,1140500,NaN
2025-05-08,544.693970,549.567368,517.671627,544.902850,2709200,NaN
2025-05-09,524.912109,549.835933,524.265617,546.016772,1499000,NaN
2025-05-12,572.502075,574.849237,532.381253,540.864931,1898300,NaN
2025-05-13,571.040039,571.925217,555.415445,568.344804,1149900,NaN
...,...,...,...,...,...,...
2026-05-01,701.419983,714.479980,700.260010,703.020020,729100,736.270647
2026-05-04,709.210022,711.549988,699.750000,702.559998,554100,733.422160
2026-05-05,702.270020,715.619995,699.229980,710.729980,642700,730.142987


In [14]:
df['EMA200'] = df['Close'].ewm(span=18, adjust=False).mean()
df

,Close,High,Low,Open,Volume,EMA50,EMA200
Date,,,,,,,
2025-05-07,557.881897,565.042784,553.336723,558.737209,1140500,NaN,557.881897
2025-05-08,544.693970,549.567368,517.671627,544.902850,2709200,NaN,556.493694
2025-05-09,524.912109,549.835933,524.265617,546.016772,1499000,NaN,553.169317
2025-05-12,572.502075,574.849237,532.381253,540.864931,1898300,NaN,555.204344
2025-05-13,571.040039,571.925217,555.415445,568.344804,1149900,NaN,556.871259
...,...,...,...,...,...,...,...
2026-05-01,701.419983,714.479980,700.260010,703.020020,729100,736.270647,736.270647
2026-05-04,709.210022,711.549988,699.750000,702.559998,554100,733.422160,733.422160
2026-05-05,702.270020,715.619995,699.229980,710.729980,642700,730.142987,730.142987


In [15]:
data = df.filter(['Close'])
data

,Close
Date,
2025-05-07,557.881897
2025-05-08,544.693970
2025-05-09,524.912109
2025-05-12,572.502075
2025-05-13,571.040039
...,...
2026-05-01,701.419983
2026-05-04,709.210022
2026-05-05,702.270020


In [16]:
# Calculate 50 and 200-day EMAs
data['EMA50'] = data['Close'].ewm(span=50, adjust=False).mean()
data['EMA200'] = data['Close'].ewm(span=200, adjust=False).mean()

# Define the crossover logic
# Signal is 1 when EMA50 > EMA200, else 0
data['Signal'] = 0.0
data.loc[data['EMA50'] > data['EMA200'], 'Signal'] = 1.0

# Find the exact crossover points
# Position is the difference in signals (1 = Golden Cross, -1 = Death Cross)
# Filter days with buy signal (==1)
data['Position'] = data['Signal'].diff()

# Extract only the dates where a crossover happened
crossovers = data[data['Position'] != 0].dropna()
print(crossovers[['Close', 'EMA50', 'EMA200', 'Position']])

                 Close       EMA50      EMA200  Position
Date                                                    
2025-05-15  581.811218  558.670818  558.051682       1.0
2025-06-04  483.007446  557.855760  558.465313      -1.0
2025-08-20  589.560730  554.022235  553.169704       1.0


In [17]:
# Filter days with buy signal
buys = crossovers[crossovers['Signal'] == 1]
print(buys.head())

                 Close       EMA50      EMA200  Signal  Position
Date                                                            
2025-05-15  581.811218  558.670818  558.051682     1.0       1.0
2025-08-20  589.560730  554.022235  553.169704     1.0       1.0


## Code for BCRX

In [18]:
import pandas as pd
import yfinance as yf
from ta.trend import EMAIndicator

favourites = ['VRTX','REGN','BCRX']
company_name = 'NA'
def closing_year(code):
    global company_name
    results = yf.download(code, period="1y", multi_level_index=False)
    ticker = yf.Ticker(code)
    # Get company information
    company_name = ticker.info.get('longName')
    return results
    
def closing_month(code):
    global company_name
    results = yf.download(code, period="1mo", multi_level_index=False)
    ticker = yf.Ticker(code)
    # Get company information
    company_name = ticker.info.get('longName')
    return results
    
# This is Fetch Data for VRTX company name
df = closing_year(favourites[2]) ## If want Know other company name can change the number for 0-5.
# df = yf.download("VRTX", period="1y", multi_level_index=False)
print(company_name)

[*********************100%***********************]  1 of 1 completed


BioCryst Pharmaceuticals, Inc.


In [19]:
df

,Close,High,Low,Open,Volume
Date,,,,,
2025-05-07,10.08,10.52,9.87,10.36,7447300
2025-05-08,9.92,10.20,9.83,10.13,10027500
2025-05-09,9.97,10.39,9.90,9.93,7607100
2025-05-12,10.34,10.50,10.07,10.14,5565700
2025-05-13,10.06,10.42,10.01,10.37,6116500
...,...,...,...,...,...
2026-05-01,9.19,9.33,9.09,9.16,2307100
2026-05-04,9.20,9.40,9.02,9.16,3454100
2026-05-05,9.03,9.31,8.94,9.22,3745200


In [20]:
df['EMA50'] = EMAIndicator(df['Close'], window=18).ema_indicator()
df

,Close,High,Low,Open,Volume,EMA50
Date,,,,,,
2025-05-07,10.08,10.52,9.87,10.36,7447300,NaN
2025-05-08,9.92,10.20,9.83,10.13,10027500,NaN
2025-05-09,9.97,10.39,9.90,9.93,7607100,NaN
2025-05-12,10.34,10.50,10.07,10.14,5565700,NaN
2025-05-13,10.06,10.42,10.01,10.37,6116500,NaN
...,...,...,...,...,...,...
2026-05-01,9.19,9.33,9.09,9.16,2307100,9.131053
2026-05-04,9.20,9.40,9.02,9.16,3454100,9.138311
2026-05-05,9.03,9.31,8.94,9.22,3745200,9.126909


In [21]:
df['EMA200'] = df['Close'].ewm(span=18, adjust=False).mean()
df

,Close,High,Low,Open,Volume,EMA50,EMA200
Date,,,,,,,
2025-05-07,10.08,10.52,9.87,10.36,7447300,NaN,10.080000
2025-05-08,9.92,10.20,9.83,10.13,10027500,NaN,10.063158
2025-05-09,9.97,10.39,9.90,9.93,7607100,NaN,10.053352
2025-05-12,10.34,10.50,10.07,10.14,5565700,NaN,10.083525
2025-05-13,10.06,10.42,10.01,10.37,6116500,NaN,10.081049
...,...,...,...,...,...,...,...
2026-05-01,9.19,9.33,9.09,9.16,2307100,9.131053,9.131053
2026-05-04,9.20,9.40,9.02,9.16,3454100,9.138311,9.138311
2026-05-05,9.03,9.31,8.94,9.22,3745200,9.126909,9.126909


In [22]:
data = df.filter(['Close'])
data

,Close
Date,
2025-05-07,10.08
2025-05-08,9.92
2025-05-09,9.97
2025-05-12,10.34
2025-05-13,10.06
...,...
2026-05-01,9.19
2026-05-04,9.20
2026-05-05,9.03


In [23]:
# Calculate 50 and 200-day EMAs
data['EMA50'] = data['Close'].ewm(span=50, adjust=False).mean()
data['EMA200'] = data['Close'].ewm(span=200, adjust=False).mean()

# Define the crossover logic
# Signal is 1 when EMA50 > EMA200, else 0
data['Signal'] = 0.0
data.loc[data['EMA50'] > data['EMA200'], 'Signal'] = 1.0

# Find the exact crossover points
# Position is the difference in signals (1 = Golden Cross, -1 = Death Cross)
# Filter days with buy signal (==1)
data['Position'] = data['Signal'].diff()

# Extract only the dates where a crossover happened
crossovers = data[data['Position'] != 0].dropna()
print(crossovers[['Close', 'EMA50', 'EMA200', 'Position']])

            Close      EMA50     EMA200  Position
Date                                             
2025-05-12  10.34  10.080259  10.079943       1.0
2025-05-13  10.06  10.079465  10.079744      -1.0
2025-05-16  10.24  10.082372  10.080381       1.0
2025-07-02   9.02  10.108425  10.126461      -1.0
2026-03-24   9.66   8.089527   8.071582       1.0


In [24]:
# Filter days with buy signal
buys = crossovers[crossovers['Signal'] == 1]
print(buys.head())

            Close      EMA50     EMA200  Signal  Position
Date                                                     
2025-05-12  10.34  10.080259  10.079943     1.0       1.0
2025-05-16  10.24  10.082372  10.080381     1.0       1.0
2026-03-24   9.66   8.089527   8.071582     1.0       1.0


## Code for Dashboard

In [25]:
import dash
from dash import html, dcc, dash_table, Input, Output
import yfinance as yf
import pandas as pd

# Download stock data for multiple tickers
tickers = ['VRTX', 'REGN', 'BCRX']
df = yf.download(tickers, period='1y')

# Prepare data for each ticker
results = []
for ticker in tickers:
    temp = df['Close'][ticker].to_frame(name='Close')
    temp['EMA50'] = temp['Close'].ewm(span=50, adjust=False).mean()
    temp['EMA200'] = temp['Close'].ewm(span=200, adjust=False).mean()
    temp['Signal'] = 0.0
    temp.loc[temp['EMA50'] > temp['EMA200'], 'Signal'] = 1.0
    temp['Position'] = temp['Signal'].diff()
    temp['Ticker'] = ticker
    results.append(temp)

# Combine all tickers
all_data = pd.concat(results)
crossovers = all_data[all_data['Position'] != 0].dropna()

# Initialize Dash app
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1("💊 Pharmaceutical Market Performance Dashboard 🧪",
    style={'fontSize': '40px', 'color': '#555', 'marginBottom': '20px'}),
    html.P(
        "📈 This dashboard provides an overview of stock performance trends using EMA50 and EMA200 crossovers "
        "to highlight potential trading signals in the healthcare sector 📉.",
        style={'fontSize': '20px', 'color': '#555', 'marginBottom': '20px'}
    ),
    html.Label("🔍 Select Ticker for Price Chart:",
               style={'color': '#64068a', 'fontWeight': 'bold'}),
    dcc.Dropdown(
        id='ticker-dropdown',
        options=[{'label': t, 'value': t} for t in tickers],
        value=tickers[0],
        clearable=False,
        style={'width': '300px', 'backgroundColor': '#a391b5', 'color': '#0914a5'}
    ),
    dcc.Graph(id='price-chart'),
    dash_table.DataTable(
        id='data-table',
        columns=[{"name": i, "id": i} for i in all_data.reset_index().columns],
        page_size=9,
        style_table={'height': '300px', 'overflowY': 'auto'},
        style_cell={'textAlign': 'left', 'padding': '5px'},
        style_header={'backgroundColor': '#e8eafc', 'fontWeight': 'bold', 'color': '#18116f'}
    )
],
        style={'backgroundColor': '#eff2f3', 'padding': '20px'})

@app.callback(
    [Output('price-chart', 'figure'),
     Output('data-table', 'data')],
    [Input('ticker-dropdown', 'value')]
)
def update_dashboard(selected_ticker):
    df_ticker = all_data[all_data['Ticker'] == selected_ticker]
    cross_ticker = crossovers[crossovers['Ticker'] == selected_ticker]

    bullish = cross_ticker[cross_ticker['Position'] == 1]
    bearish = cross_ticker[cross_ticker['Position'] == -1]

    figure = {
        "data": [
            {
                "x": df_ticker.index, 
                "y": df_ticker["Close"], 
                "type": "line", 
                "name": "Close", 
                "line": {"color": "black"}
            },
            {
                "x": df_ticker.index, 
                "y": df_ticker["EMA50"], 
                "type": "line", 
                "name": "EMA50", 
                "line": {"color": "blue", 
                         "dash": "dot"}
            },
            {
                "x": df_ticker.index, 
                "y": df_ticker["EMA200"], 
                "type": "line", 
                "name": "EMA200", 
                "line": {"color": "red", 
                         "dash": "dot"}},
            {
                "x": bullish.index,
                "y": bullish["Close"],
                "mode": "markers",
                "name": "Bullish Crossovers",
                "marker": {"color": "green", "size": 10, "symbol": "circle"},
            },
            {
                "x": bearish.index,
                "y": bearish["Close"],
                "mode": "markers",
                "name": "Bearish Crossovers",
                "marker": {"color": "red", "size": 10, "symbol": "circle"},
            },
        ],
        "layout": {
            "title": {
                "text": f"💹<b>{selected_ticker}</b> Close Price with EMA50 & EMA200 Crossovers",
                "font": {"size": 20}
            },
            "xaxis": {"title": {"text": "<b>Date</b>"}},
            "yaxis": {"title": {"text": "<b>Price</b>"}},
        },
    }

    table_data = df_ticker.reset_index().to_dict('records')
    return figure, table_data



if __name__ == '__main__':
    app.run(jupyter_mode='external', port=8082)


[*********************100%***********************]  3 of 3 completed


Dash app running on http://127.0.0.1:8082/
